## Init

In [7]:
import json
import logging
import operator
import os
import re
import sqlite3
import textwrap
from datetime import date, datetime, time
from functools import partial
from pathlib import Path
from typing import Annotated, Any, Optional, Sequence

import altair as alt
import numpy as np
import polars as pl
import torch
from IPython.utils.capture import capture_output
from polars import col as c
from polars import lit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from tqdm import tqdm
from transformers import AutoTokenizer

# Configure logging to show timestamp, log level and message
logging.basicConfig(
    format="[%(asctime)s] [%(levelname)s] %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S %p",
)
# Add these lines to suppress HTTP request logs
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("requests").setLevel(logging.WARNING)


# set working directory
def set_wdir(wdir) -> tuple[Path, Path]:
    # get working directory
    if isinstance(wdir, str):
        wdir = Path(wdir).expanduser()
    chaoyang_dir = Path("~/chaoyang").expanduser()

    # assert directories exist
    assert wdir.exists()
    assert chaoyang_dir.exists()

    # change working directory
    os.chdir(wdir)

    return wdir, chaoyang_dir


with capture_output():
    # initialize polars
    alt.data_transformers.enable("vegafusion")
    pl.Config.set_tbl_rows(20)  # max number of rows to print

# initialize directories
work_dir, chaoyang_dir = set_wdir("~/chaoyang/projects/Delinquency/delinquency/github/v2")
print(f"WORK DIR: {work_dir}")
print(f"CHAOYANG DIR: {chaoyang_dir}")

WORK DIR: /home/yuzhu/chaoyang/projects/Delinquency/delinquency/github/v2
CHAOYANG DIR: /home/yuzhu/chaoyang


## Data Exploration

In [33]:
# load sample index
samples = pl.read_ipc("data/processed_data/llm_benchmark/samples_min12mo_fixed_2test.feather", memory_map=False)

# load transaction data
transactions = pl.read_ipc("data/raw_data/sample_transaction.feather", memory_map=False)

# how many users? (3178)
print(f"Number of users: {samples['act_idn_sky'].n_unique()}")

# sample size (number of rolling windows)?
print(f"# Obs in Test: {samples.filter(c.split == 'test').height:,}")
print(f"# Obs in Train: {samples.filter(c.split == 'train').height:,}")

# average number of billing cycles for a user?
act_idn_sky = samples["act_idn_sky"].unique().to_list()
avg_n_cycles = (
    transactions.filter(c.act_idn_sky.is_in(act_idn_sky))
    .group_by("act_idn_sky")
    .agg(n_cycles = c.billing_date.n_unique())
    .select(
        avg_n_cycles = c.n_cycles.mean(),
        max_n_cycles = c.n_cycles.max(),
        min_n_cycles = c.n_cycles.min(),
    )
)
print("Summary of billing cycle numbers per user:")
avg_n_cycles


avg_n_txn = (
    transactions.filter(c.act_idn_sky.is_in(act_idn_sky))
    .group_by("act_idn_sky")
    .agg(n_txn=pl.len())
    .select(
        avg_n_txn = c.n_txn.mean(),
        max_n_txn = c.n_txn.max(),
        min_n_txn = c.n_txn.min(),
    )
)
print("Summary of transaction numbers per user:")
avg_n_txn

Number of users: 3178
# Obs in Test: 5,363
# Obs in Train: 17,223
Summary of billing cycle numbers per user:


avg_n_cycles,max_n_cycles,min_n_cycles
f64,u32,u32
18.106986,30,12


Summary of transaction numbers per user:


avg_n_txn,max_n_txn,min_n_txn
f64,u32,u32
88.232536,1206,12


## Evaluation

### LLM-based

Collect Prediction Results

In [8]:
# function to parse `pred_is_delinquent_raw`
def parse_is_delinquent_raw(s):
    pat = re.compile(r'\{[^{}]*"is_delinquent"\s*:\s*(true|false)[^{}]*\}', re.I | re.S)

    if not isinstance(s, str):
        return None

    m = pat.search(s)
    if not m:
        return None

    try:
        obj = json.loads(m.group(0).replace("True", "true").replace("False", "false"))
        return obj.get("is_delinquent") if isinstance(obj.get("is_delinquent"), bool) else None
    except json.JSONDecodeError:
        return None


# get paths of all preds tables
preds_paths = sorted(list((work_dir / "data/processed_data/llm_benchmark").glob("preds_*.feather")))

# merge all preds tables into one
preds = []
check_cols = ["pred_reasoning_process", "config_max_prompts"]

for i, preds_path in enumerate(preds_paths):
    
    # read preds table
    samples = pl.read_ipc(preds_path, memory_map=False)

    # if has `pred_is_delinquent_raw` column, parse it
    if "pred_is_delinquent_raw" in samples.columns:

        # if `pred_is_delinquent` is not in samples, create it
        if "pred_is_delinquent" not in samples.columns:
            samples = samples.with_columns(pred_is_delinquent=None)

        samples = (
            samples
            # parse `pred_is_delinquent_raw`
            .with_columns(
                pred_is_delinquent_raw_parsed=c.pred_is_delinquent_raw.map_elements(
                    parse_is_delinquent_raw, return_dtype=pl.Boolean
                )
            )
            # update `pred_is_delinquent` with `pred_is_delinquent_raw`
            .with_columns(pred_is_delinquent=pl.coalesce(c.pred_is_delinquent, c.pred_is_delinquent_raw_parsed))
        )

        # drop `pred_is_delinquent_raw` column
        samples = samples.drop("pred_is_delinquent_raw")

        # check if any `pred_is_delinquent_raw` failed to parse
        failures = samples.filter(c.pred_is_delinquent.is_null())
        if failures.height > 0:
            print(f"Warning: {failures.height} rows failed to parse ({preds_path.stem})")

        # drop rows where `pred_is_delinquent` is null
        samples = samples.filter(c.pred_is_delinquent.is_not_null())

    if "pred_a4_output" in samples.columns:
        samples = samples.with_columns(
            pred_is_delinquent=pl.when(c.pred_a4_output.struct.field("is_delinquent").str.to_lowercase()=="true").then(True).otherwise(False),
        )

    # add experiment ID
    exp_id = preds_path.stem
    samples = samples.with_columns(exp_id=lit(exp_id))

    # add benchmark name (e.g., finpt, promptcast, etc.)
    benchmark_name = preds_path.stem.split("_")[1]
    samples = samples.with_columns(benchmark_name=lit(benchmark_name))
    preds.append(samples)

# concat all preds tables
preds = pl.concat(preds, how="diagonal_relaxed")

# if sys_msg is null (e.g., finpt), fill it with empty string
preds = preds.with_columns(c.sys_msg.fill_null(""))

# reorder columns
col_names = sorted(preds.columns)
preds = preds.select(col_names)

# check if N obs per experiment is correct
nobs_error = (
    preds["exp_id"].value_counts().sort(c.exp_id)
    .filter(~c.count.is_in([5363,10020,5362, 2841]))
)
if nobs_error.height > 0:
    print("Warning: N obs per experiment incorrect for:")
    nobs_error

In [9]:
preds[:1]

act_idn_sky,benchmark_name,billing_dates,birth_year,config_analyze_token_len,config_embeddings_filename,config_has_protected_attributes,config_is_cot_prompt,config_llm_name,config_max_completion_tokens,config_max_ctx_size,config_max_prompts,config_max_transaction_tokens,config_max_workers,config_sample_path,config_transaction_text_type,config_use_transaction_summary,cumulative_delin_times,delinquency_history,education,exp_id,industry,lvl_4_bch_nam,marriage_status,n_txn,pred_delinquency_prob,pred_embed,pred_is_delinquent,pred_is_delinquent_raw_parsed,pred_logprobs,pred_prob,pred_reasoning_process,pred_risk_level,residence,sex,split,sys_msg,target_delinquency,total_amt,transaction_summary,transaction_text,transaction_text_agent_detail_1,transaction_text_detail_1,transaction_text_detail_2,transaction_text_detail_3,transaction_text_finpt_detail_1,transaction_text_finpt_summary_1,transaction_text_promptcast_detail_1,transaction_text_promptcast_summary_1,transaction_text_summary_0,transaction_text_summary_1,transaction_text_summary_2,transaction_text_summary_3,user_msg
str,str,list[date],i64,bool,str,bool,bool,str,i32,i32,null,i32,i32,str,str,bool,i64,list[bool],str,str,str,str,str,list[i64],f64,"array[f32, 3584]",bool,bool,list[struct[3]],f32,str,str,str,str,str,str,bool,list[f64],str,str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""001A000600009658""","""finpt""","[2012-02-21, 2012-03-21, … 2013-01-21]",1982,null,"""embeds_finpt_detail_1_20251205…",null,null,"""qwen2.5-7b-instruct""",null,null,null,15000,null,"""llm_benchmark/samples_min12mo_…","""finpt_detail_1""",null,0,"[false, false, … false]","""未知""","""preds_finpt_detail_1_20251205_…","""未知""","""江苏分行""","""单身""","[16, 16, … 11]",null,"[-0.000126, 0.002278, … 0.001099]",false,null,null,0.012619,null,null,"""未知""","""女""","""test""","""""",false,"[7360.0, 5271.0, … 2508.0]",null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""【信用卡用户信息】： - 居住地：南京 - 所属分支行为：江…"


Run Evaluation

In [12]:
class BinaryClassificationEvaluator:
    """Evaluator for binary classification predictions with comprehensive metrics."""

    def __init__(
        self,
        data,
        target_col="target",
        pred_col="pred",
        prob_col=None,
        threshold=0.5,
        get_token_length=False,
        print_prompt=False,
    ):
        """
        Initialize the evaluator.

        Args:
            data: Polars DataFrame with target, pred, and optionally prob columns
            target_col: Name of the target column (default: "target")
            pred_col: Name of the prediction column (default: "pred")
            prob_col: Name of the probability column (default: None, optional)
            threshold: Threshold for binary classification when using probabilities (default: 0.5)
        """
        self.data = data
        self.target_col = target_col
        self.pred_col = pred_col
        self.prob_col = prob_col
        self.threshold = threshold
        self.print_prompt = print_prompt
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-30B-A3B-Thinking-2507")

        # Extract arrays
        self.y_true = data[target_col].to_numpy()
        self.y_pred = data[pred_col].to_numpy()

        # Calculate class distribution
        self.majority_class = int(self.y_true.mean() > 0.5)
        self.class_counts = pl.DataFrame({target_col: self.y_true}).group_by(target_col).len()
        self.total = len(self.y_true)

        # Check if probability column is available
        self.has_prob = (
            prob_col is not None
            and prob_col in data.columns
            and data[prob_col].null_count() < len(data)
        )

        # get average token length
        if get_token_length:
            self.avg_token_length = self.calcualte_token_length()
        else:
            self.avg_token_length = "Not calc"

    def calcualte_token_length(self):
        """Calculate the token length of a list of messages."""
        # get the messages
        messages = self.data["sys_msg"] + self.data["user_msg"]
        messages = messages.to_list()

        # Get the length of each message in tokens
        tokenized = self.tokenizer(messages, padding=False, truncation=False)
        lengths = [len(ids) for ids in tokenized["input_ids"]]

        # get average length
        return f"{np.mean(lengths):.0f}"

    def calculate_metrics(self):
        """Calculate all evaluation metrics."""
        # Calculate baseline metrics
        y_baseline = [self.majority_class] * len(self.y_true)
        self.baseline_metrics = {
            "accuracy": accuracy_score(self.y_true, y_baseline),
            "precision": precision_score(self.y_true, y_baseline, zero_division=0),
            "recall": recall_score(self.y_true, y_baseline, zero_division=0),
            "f1": f1_score(self.y_true, y_baseline, zero_division=0),
            "auc": None,
            "aupr": None,
        }

        # Calculate model metrics - use threshold-based predictions if prob_col and threshold are available
        if self.has_prob and self.threshold is not None:
            y_prob = self.data[self.prob_col].to_numpy()
            y_pred_threshold = (y_prob >= self.threshold).astype(int)

            self.metrics = {
                "accuracy": accuracy_score(self.y_true, y_pred_threshold),
                "precision": precision_score(self.y_true, y_pred_threshold, zero_division=0),
                "recall": recall_score(self.y_true, y_pred_threshold, zero_division=0),
                "f1": f1_score(self.y_true, y_pred_threshold, zero_division=0),
            }
        else:
            # Use original predictions
            self.metrics = {
                "accuracy": accuracy_score(self.y_true, self.y_pred),
                "precision": precision_score(self.y_true, self.y_pred, zero_division=0),
                "recall": recall_score(self.y_true, self.y_pred, zero_division=0),
                "f1": f1_score(self.y_true, self.y_pred, zero_division=0),
            }

        # Calculate probability-based metrics if available
        if self.has_prob:
            y_prob = self.data[self.prob_col].to_numpy()

            try:
                self.metrics["auc"] = roc_auc_score(self.y_true, y_prob)
            except (ValueError, TypeError) as e:
                print(f"Warning: Could not calculate AUC - {e}")
                self.metrics["auc"] = None

            try:
                self.metrics["aupr"] = average_precision_score(self.y_true, y_prob)
            except (ValueError, TypeError) as e:
                print(f"Warning: Could not calculate AUPR - {e}")
                self.metrics["aupr"] = None
        else:
            self.metrics["auc"] = None
            self.metrics["aupr"] = None

        return {
            "model": self.metrics,
            "baseline": self.baseline_metrics,
            "majority_class": self.majority_class,
        }

    def print_results(self):
        """Print evaluation results in a formatted table."""
        if self.print_prompt:
            print(f"[SYSTEM MESSAGE]:\n\n{self.data['sys_msg'][0]}")
            print("-" * 60)
            print()
            print(
                f"[USER MESSAGE Example]:\n\n{'\n'.join(self.data['user_msg'][0].splitlines()[:15])}"
            )
            print("...")

        print("\n" + "=" * 70)
        print("BINARY CLASSIFICATION EVALUATION RESULTS")
        print("=" * 70)

        # print experiment metadata
        exp_id = self.data["exp_id"][0]
        sample_name = Path(self.data["config_sample_path"][0]).stem
        has_protected = self.data["config_has_protected_attributes"][0]
        is_cot_prompt = self.data["config_is_cot_prompt"][0]
        llm_name = self.data["config_llm_name"][0]
        txn_text_type = self.data["config_transaction_text_type"][0]
        print(f"\nExperiment ID: {exp_id}")
        print(f"  Sample: {sample_name}")
        print(f"  Transaction Text Type: {txn_text_type}")
        print(f"  Avg Token Length: {self.avg_token_length}")
        print(f"  Is COT Prompt: {is_cot_prompt}")
        print(f"  Has Protected: {has_protected}")
        print(f"  LLM: {llm_name}")

        # Print class distribution
        print("\nClass Distribution:")
        print(f"  Total: {self.data.height:,}")
        for row in self.class_counts.iter_rows(named=True):
            class_val = row[self.class_counts.columns[0]]
            count = row["len"]
            pct = count / self.total * 100
            marker = " (MAJORITY)" if class_val == self.majority_class else ""
            print(f"  Class {class_val}: {count:,} ({pct:.2f}%){marker}")

        # Print model metrics
        print("\nModel Performance:")
        if self.has_prob and self.threshold is not None:
            print(f"  (Using threshold: {self.threshold})")
        print(f"  F1 Score:  {self.metrics['f1']:.4f}")
        print(f"  Recall:    {self.metrics['recall']:.4f}")
        print(f"  Precision: {self.metrics['precision']:.4f}")
        print(f"  Accuracy:  {self.metrics['accuracy']:.4f}")

        # Print probability-based metrics if available
        if self.has_prob and self.metrics.get("auc") is not None:
            print(f"  AUC:       {self.metrics['auc']:.4f}")
            print(f"  AUPR:      {self.metrics['aupr']:.4f}")

        print("=" * 70 + "\n")

    def evaluate(self):
        """Run full evaluation: calculate metrics and print results."""
        results = self.calculate_metrics()
        self.print_results()
        # return results


evaluator = BinaryClassificationEvaluator(
    data=(
        preds
        .filter(
            c.exp_id == "preds_promptcast_summary_1_20251204_181833",
            # c.delinquency_history.list.last() == 1,
            # c.delinquency_history.list.sum() <= 5
        )
    ),
    target_col="target_delinquency",
    pred_col="pred_is_delinquent",
    prob_col=None,
    # threshold=0.5,
    get_token_length=True,
    print_prompt=True,
)
evaluator.evaluate()

[SYSTEM MESSAGE]:

角色：你是资深金融风控建模专家，熟悉信用评分、逾期定义、卡账行为、稳定性检验。

任务：基于给定用户的个人信息和信用卡历史数据，预测其在最新一个账单周期是否能按时还款。只使用提供的数据，不要臆测缺失值。

请输出合法JSON格式（不包含 markdown code block），包含以下字段：
{
    "is_delinquent": boolean
}
------------------------------------------------------------

[USER MESSAGE Example]:

【用户信息】：
- 居住地：南京
- 所属分支行为：江苏分行
- 居住情况：未知
- 行业：未知
- 学历：未知

【消费与历史违约情况】：
该用户在过去12个账单周期中，共发生0次违约。

在2012年2月的账单周期，总支出金额为7360.0元，共发生16笔交易，

在2012年3月的账单周期，总支出金额为5271.0元，共发生16笔交易，

在2012年4月的账单周期，总支出金额为14843.0元，共发生26笔交易，
...

BINARY CLASSIFICATION EVALUATION RESULTS

Experiment ID: preds_promptcast_summary_1_20251204_181833
  Sample: samples_min12mo_fixed_2test
  Transaction Text Type: promptcast_summary_1
  Avg Token Length: 545
  Is COT Prompt: False
  Has Protected: False
  LLM: qwen2.5-7b-instruct

Class Distribution:
  Total: 5,363
  Class False: 4,924 (91.81%) (MAJORITY)
  Class True: 439 (8.19%)

Model Performance:
  F1 Score:  0.4740
  Recall:    0.5194
  Precision: 0.4359
  Accuracy:  0.9056



In [79]:
# old
preds_old = (
    preds.filter(c.exp_id == "preds_20251018_022543")
    .select(c.act_idn_sky, c.billing_dates, c.sys_msg, c.user_msg, c.pred_is_delinquent)
)
preds_new = (
    preds.filter(c.exp_id == "preds_20251128_030635")
    .select(c.act_idn_sky, c.billing_dates, c.sys_msg, c.user_msg, c.pred_is_delinquent)
)

df = preds_old.join(preds_new, on=["act_idn_sky", "billing_dates"], how="full")

In [80]:
print(preds_new[:1]['user_msg'].item())

【用户信息】：
- 居住地：南京
- 所属分支行为：江苏分行
- 居住情况：未知
- 行业：未知
- 学历：未知

【消费与历史违约情况】：
你过去的消费与违约情况如下：
所有用户平均逾期率(delinquency rate)为7.61%

在2012年2月的账单周期，应付账单金额为81551元，共发生16笔交易，该账单周期是否违约：否，具体消费如下：
时间：2012-01-29 20:12:04，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:15:19，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:19:02，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:21:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:25:12，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:32:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:35:21，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:38:03，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:41:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-02-02 10:31:26，商户类别：航空公司，交易描述：华程西南旅行社，金额960.0元。
时间：2012-02-06 11:51:21，商户类别：航空公司，交易描述：华程西南旅行社，金额440.0元。
时间：2012-02-09 08:36:30，商户类别：慈善和社会公益服务组织，交易描述：中国银联，金额100.0元。
时间：2012-02-12 18:18:32，商户类别：航空公司，交易描述：中国东方航空股份有限，金额520.0元。
时间：2012-02-15 12:54:56，商户类别：住宿服务（旅馆），交易描述：明捷置业，金额1439.97元。
时

In [46]:
with pl.Config(fmt_str_lengths=100):
    df.select(c.user_msg, c.user_msg_right).sample()

user_msg,user_msg_right
str,str
"""【用户信息】： - 居住地：南京 - 所属分支行为：None - 居住情况：无按揭自置 - 行业：能源 - 学历：大学本科 【消费与历史违约情况】： 你过去的消费与违约情况如下： 在2012年3月的…","""【用户信息】： - 居住地：南京 - 所属分支行为：None - 居住情况：无按揭自置 - 行业：能源 - 学历：大学本科 【消费与历史违约情况】： 你过去的消费与违约情况如下： 在2012年3月的…"


In [26]:
# print(preds_old[:1]['sys_msg'].item())
print(preds_old[:1]['user_msg'].item())

【用户信息】：
- 居住地：南京
- 所属分支行为：江苏分行
- 居住情况：未知
- 行业：未知
- 学历：未知

【消费与历史违约情况】：
你过去的消费与违约情况如下：
在2012年2月的账单周期，应付账单金额为81551元，该账单周期是否违约：否，具体消费如下：
时间：2012-01-29 20:12:04，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:15:19，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:19:02，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:21:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:25:12，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:32:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:35:21，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:38:03，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-01-29 20:41:45，商户类别：慈善和社会公益服务组织，交易描述：支付宝，金额300.0元。
时间：2012-02-02 10:31:26，商户类别：航空公司，交易描述：华程西南旅行社，金额960.0元。
时间：2012-02-06 11:51:21，商户类别：航空公司，交易描述：华程西南旅行社，金额440.0元。
时间：2012-02-09 08:36:30，商户类别：慈善和社会公益服务组织，交易描述：中国银联，金额100.0元。
时间：2012-02-12 18:18:32，商户类别：航空公司，交易描述：中国东方航空股份有限，金额520.0元。
时间：2012-02-15 12:54:56，商户类别：住宿服务（旅馆），交易描述：明捷置业，金额1439.97元。
时间：2012-02-16 07:52:23，商户类别：住宿服务（旅馆），交易描述：中油天

### Logistic

In [9]:
def calc_metrics(x_train, y_train, x_test, y_test):

    # Train logistic regression model
    lr_model = LogisticRegression(max_iter=1000)
    lr_model.fit(x_train, y_train)

    # Predict on test set
    y_pred = lr_model.predict(x_test)

    # Calculate metrics (with zero_division=0 to suppress warnings)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    accuracy = accuracy_score(y_test, y_pred)

    print(f"Logistic Regression Results on Test Set:")
    print(f"  N: {len(y_test)}")
    print(f"  F1 Score: {f1:.4f}")
    print(f"  Recall: {recall:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Accuracy: {accuracy:.4f}")

delin = cum_delin

In [15]:
# load sample index
samples = pl.read_ipc("data/processed_data/llm_benchmark/samples_min12mo_fixed_2test.feather", memory_map=False)

# Use cum_delin as features
x_test = samples.filter(c.split == "test").select(c.cumulative_delin_times).to_numpy().reshape(-1, 1)
y_test = samples.filter(c.split == "test").select(c.target_delinquency).to_numpy().ravel()
x_test_firsttime = (
    samples
    .filter(c.split == "test", c.delinquency_history.list.last(), c.delinquency_history.list.sum() <= 3)
    .select(c.cumulative_delin_times)
    .to_numpy()
    .reshape(-1, 1)
)
y_test_firsttime = (
    samples.filter(c.split == "test", c.delinquency_history.list.last(), c.delinquency_history.list.sum() <= 3)
    .select(c.target_delinquency)
    .to_numpy()
    .ravel()
)

x_train = samples.filter(c.split == "train").select(c.cumulative_delin_times).to_numpy().reshape(-1, 1)
y_train = samples.filter(c.split == "train").select(c.target_delinquency).to_numpy().ravel()

print("=" * 80)
print("ALL TEST SAMPLES")
print("=" * 80)
calc_metrics(x_train, y_train, x_test, y_test)

print("\n" + "=" * 80)
print("FIRST-TIME DELINQUENCY TEST SAMPLES")
print("=" * 80)
calc_metrics(x_train, y_train, x_test_firsttime, y_test_firsttime)
print("=" * 80)

ALL TEST SAMPLES
Logistic Regression Results on Test Set:
  N: 5363
  F1 Score: 0.4259
  Recall: 0.3144
  Precision: 0.6603
  Accuracy: 0.9306

FIRST-TIME DELINQUENCY TEST SAMPLES
Logistic Regression Results on Test Set:
  N: 201
  F1 Score: 0.0000
  Recall: 0.0000
  Precision: 0.0000
  Accuracy: 0.0000


delin = cum_delin + total_amt + N txn

In [99]:
# load sample index
samples = pl.read_ipc("data/processed_data/llm_benchmark/samples_min12mo_fixed_2test.feather", memory_map=False)

# Use cum_delin as features
data = (
    samples
    .select(c.split, c.delinquency_history, c.cumulative_delin_times, c.total_amt, c.n_txn, c.target_delinquency)
    .with_columns(c.total_amt.list.to_struct(fields=[f"total_amt_{i}" for i in range(12)]))
    .with_columns(c.n_txn.list.to_struct(fields=[f"n_txn_{i}" for i in range(12)]))
    .unnest("total_amt")
    .unnest("n_txn")
)

x_test = (
    data
    .filter(c.split == 'test')
    .select(c.cumulative_delin_times, pl.col([f"total_amt_{i}" for i in range(12)]), pl.col([f"n_txn_{i}" for i in range(12)]))
    .to_numpy()
)
y_test = (
    data
    .filter(c.split == 'test')
    .select(c.target_delinquency)
    .to_numpy()
    .ravel()
)

# First-time delinquency test samples
x_test_firsttime = (
    data
    .filter(c.split == 'test', c.delinquency_history.list.last(), c.delinquency_history.list.sum()<=2)
    .select(c.cumulative_delin_times, pl.col([f"total_amt_{i}" for i in range(12)]), pl.col([f"n_txn_{i}" for i in range(12)]))
    .to_numpy()
)
y_test_firsttime = (
    data
    .filter(c.split == 'test', c.delinquency_history.list.last(), c.delinquency_history.list.sum()<=2)
    .select(c.target_delinquency)
    .to_numpy()
    .ravel()
)

x_train = (
    data
    .filter(c.split == 'train')
    .select(c.cumulative_delin_times, pl.col([f"total_amt_{i}" for i in range(12)]), pl.col([f"n_txn_{i}" for i in range(12)]))
    .to_numpy()
)
y_train = (
    data
    .filter(c.split == 'train')
    .select(c.target_delinquency)
    .to_numpy()
    .ravel()
)

print("=" * 80)
print("MULTI-FEATURE LOGISTIC REGRESSION - ALL TEST SAMPLES")
print("=" * 80)
calc_metrics(x_train, y_train, x_test, y_test)

print("\n" + "=" * 80)
print("MULTI-FEATURE LOGISTIC REGRESSION - FIRST-TIME DELINQUENCY")
print("=" * 80)
calc_metrics(x_train, y_train, x_test_firsttime, y_test_firsttime)
print("=" * 80)

MULTI-FEATURE LOGISTIC REGRESSION - ALL TEST SAMPLES


/home/yuzhu/App/python-env/py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic Regression Results on Test Set:
  F1 Score: 0.4106
  Recall: 0.3166
  Precision: 0.5840
  Accuracy: 0.9256

MULTI-FEATURE LOGISTIC REGRESSION - FIRST-TIME DELINQUENCY
Logistic Regression Results on Test Set:
  F1 Score: 0.0000
  Recall: 0.0000
  Precision: 0.0000
  Accuracy: 0.0000


/home/yuzhu/App/python-env/py312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:470: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Sketch

In [3]:
# load sample index
samples = pl.read_ipc("data/processed_data/llm_benchmark/samples_min12mo_fixed_2test.feather", memory_map=False)
sample_index = pl.read_ipc("data/processed_data/sample_index/index_min12mo_fixed_2test.feather", memory_map=False)

# load transaction data
transactions = pl.read_ipc("data/raw_data/sample_transaction.feather", memory_map=False)

# load prediction results
preds = pl.read_ipc("data/processed_data/llm_benchmark/preds_promptcast_detail_1_20251205_020652.feather", memory_map=False)

# load embeddings
# emb34 = torch.load("data/processed_data/llm_benchmark/embeds_finpt_detail_1_20251204_155334.pt")
# emb51 = torch.load("data/processed_data/llm_benchmark/embeds_finpt_detail_1_20251204_214351.pt")

In [6]:
preds[:1]['pred_is_delinquent_raw'].item()

'```json\n{\n    "is_delinquent": false\n}\n```'

In [26]:
sum([1 for ele in emb34 if ele['embedding'] is None])
sum([1 for ele in emb51 if ele['embedding'] is None])


10

22586

In [8]:
len(emb34)
sum([(e1['embedding'] == e2['embedding']).all() for e1, e2 in zip(emb34, emb43)])


22586

tensor(22586)

In [53]:
emb43[987]['embedding']
emb34[987]['embedding']

tensor([-0.0011, -0.0023, -0.0030,  ..., -0.0015, -0.0012, -0.0011])

tensor([-0.0011, -0.0023, -0.0030,  ..., -0.0015, -0.0012, -0.0011])

In [90]:
with pl.Config(set_fmt_table_cell_list_len=1000):
    (
        samples
        .filter(c.split == 'test')
        .with_columns(c.delinquency_history.list.eval(pl.element().cast(pl.Int64)))
        # find the delinquent samples
        .filter(c.delinquency_history.list.last()==1)
        # delinquency 1 time or less
        # .filter(c.delinquency_history.list.sum()<=2)
        ['delinquency_history'].value_counts(sort=True)
        .with_columns(proportion=(c.count/c.count.sum()).round(4), n=c.count.sum())
        # get first-time delinquency
        # .filter(c.delinquency_history==[0,0,0,0,0,0,0,0,0,0,0,1])
    )

delinquency_history,count,proportion,n
list[i64],u32,f64,u32
"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",70,0.1595,439
"[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1]",9,0.0205,439
"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]",8,0.0182,439
"[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1]",7,0.0159,439
"[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1]",7,0.0159,439
"[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1]",7,0.0159,439
"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",6,0.0137,439
"[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]",5,0.0114,439
"[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1]",5,0.0114,439
